In [6]:
import os
import requests
import pandas as pd

os.makedirs('../data/external', exist_ok=True)
output_csv = '../data/external/nyc_cscl_road_types.csv'

if not os.path.exists(output_csv):
    print("Querying SODA v3 endpoint for NYC Street Centerline roadway classifications...")
    
    url = "https://data.cityofnewyork.us/api/v3/views/inkn-q76z/query.json"
    
    # SoQL query: group by street name and roadway type to pull clean distinct rows
    # rw_type: 1 = Street/Local, 2 = Highway/Major Arterial, 3 = Bridge, 4 = Tunnel, 6 = Ramp
    query = """
    SELECT 
        `st_name`, 
        `rw_type`, 
        count(*) as count 
    WHERE `st_name` IS NOT NULL AND `rw_type` IS NOT NULL 
    GROUP BY `st_name`, `rw_type`
    """
    
    params = {
        "query": query
    }
    headers = {
        "User-Agent": "CUNY-Columbia-NYU-Capstone/1.0"
    }
    
    resp = requests.get(url, params=params, headers=headers, timeout=60)
    
    # If GET returns method restriction, retry with POST (standard in SODA v3)
    if resp.status_code != 200:
        resp = requests.post(url, json={"query": query}, headers=headers, timeout=60)
        
    resp.raise_for_status()
    data = resp.json()
    
    # Parse rows from response
    rows = data.get('rows', data) if isinstance(data, dict) else data
    df = pd.DataFrame(rows)
    
    df['st_name'] = df['st_name'].astype(str).str.strip().str.upper()
    df['rw_type'] = pd.to_numeric(df['rw_type'], errors='coerce')
    
    # Deduplicate street name by taking its highest-capacity classification
    # (if an avenue has both highway and local segments, prioritize highway/arterial)
    df_sorted = df.sort_values(by=['st_name', 'rw_type'], ascending=[True, False])
    cscl_unique = df_sorted.drop_duplicates(subset=['st_name'], keep='first')
    
    cscl_unique.to_csv(output_csv, index=False)
    print(f"Success! Saved {len(cscl_unique):,} street classifications to {output_csv}")
    print(cscl_unique.head())
else:
    print(f"Found cached street types at {output_csv}")
    cscl_unique = pd.read_csv(output_csv)
    print(cscl_unique.head())

Querying SODA v3 endpoint for NYC Street Centerline roadway classifications...


HTTPError: 403 Client Error: Forbidden for url: https://data.cityofnewyork.us/api/v3/views/inkn-q76z/query.json